In [1]:
import pandas as pd
from functools import reduce
import ast
import numpy as np
import json
from pathlib import Path
from config import RESULTS_DIR

2026-01-29 16:52:51.065 | INFO     | config:<module>:6 - PROJ_ROOT path is: /home/navarri/AtriaProject/deepcsdf-atria


In [2]:
def parse_value(x):
    # numpy array or list from LDDMM
    if isinstance(x, (np.ndarray, list, tuple)):
        return float(x[0])
    # scalar from chamfer
    return float(x)

In [ ]:
fname = RESULTS_DIR / "metrics" / "version_0-LDDMM-trainshapes.parquet"
df = pd.read_parquet(fname)
df["value"] = df["value"].apply(parse_value)
df

Mean error per version, per organ, across patients

In [ ]:
df.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

Mean error per version, across organs, across patients : one single number per version. Organs are weighted the same here.

In [ ]:
df.groupby(["version", "metric"])["value"].agg(
    mean="mean",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

# Load several versions / metrics

In [21]:
# retrieve all the wanted files
dfs = []

for file_path in Path("results/metrics").glob("version_*.parquet"):
    # extract version from filename, e.g. version_89.csv → 89
    version = file_path.stem.split("-")[0].split("_")[-1]
    df = pd.read_parquet(file_path)
    df["version"] = int(version)
    df["value"] = df["value"].apply(parse_value)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
# df_across_vers["version"].unique() # inspect which versions are there
df_all

/tmp/ipykernel_3925478/3287992365.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(x)
/tmp/ipykernel_3925478/3287992365.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(x)
/tmp/ipykernel_3925478/3287992365.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(x)


,version,patient,organ,metric,value
0,114,AF059,epicardium,LDDMM,0.012995
1,114,AF059,la_endo,LDDMM,0.012610
2,114,AF059,ra_endo,LDDMM,0.001626
3,114,LEU_NORM_F017,epicardium,LDDMM,0.004664
4,114,LEU_NORM_F017,la_endo,LDDMM,0.005728
...,...,...,...,...,...
175,100,AF037,la_endo,LDDMM,1.023166
176,100,AF037,ra_endo,LDDMM,0.016999
177,100,LEU_NORM_0717,epicardium,LDDMM,0.038645
178,100,LEU_NORM_0717,la_endo,LDDMM,0.006885


In [22]:
df_all.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

mean    median       std       q95
version metric  organ                                             
89      LDDMM   epicardium  0.050874  0.034255  0.053713  0.136110
                la_endo     0.124234  0.020904  0.328165  0.604930
                ra_endo     0.017285  0.012245  0.015116  0.043434
        chamfer epicardium  0.038532  0.034915  0.012231  0.057736
                la_endo     0.050239  0.042567  0.015669  0.076686
                ra_endo     0.049449  0.042124  0.015545  0.072878
100     LDDMM   epicardium  0.074344  0.064056  0.059302  0.165759
                la_endo     0.111134  0.007143  0.320532  0.575388
                ra_endo     0.015760  0.008786  0.015041  0.041536
        chamfer epicardium  0.039617  0.040101  0.009228  0.052566
                la_endo     0.048519  0.050083  0.009943  0.060854
                ra_endo     0.052022  0.054606  0.012100  0.068549
114     LDDMM   epicardium  0.022527  0.010189  0.025852  0.065143
                la_endo     0.114323  0.009169  0.313919  0.578367
                ra_endo     0.017210  0.002941  0.029987  0.072074
        chamfer epicardium  0.027821  0.027305  0.010946  0.040410
                la_endo     0.037169  0.038756  0.014380  0.056875
                ra_endo     0.041515  0.032983  0.027775  0.085856

Fr example: keep chamfer only, per version, per patient, per organ

In [ ]:
df_chamfer = df_all.query(" metric == 'chamfer' ")
df_chamfer

In [ ]:
chamfer_summary = df_chamfer.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)
chamfer_summary

Polish the table to then have in latex

In [ ]:
chamfer_summary = chamfer_summary.reset_index().drop(columns=['metric'])

# multirow needs manual edits (like not repeating version number for every organ row ...), but for simple tables:
latex_table = chamfer_summary.to_latex(index=False)
print(latex_table)

# Ranking versions
Have a single number basically summarizing the performancefor a particular metric across patients, across organs, for each version. Ranking to remove anatomy "difficulty", scale of the metric (...)

In [29]:
df_chamfer_epi = df_all.query(" metric == 'chamfer' and organ == 'epicardium' ")
df_LDDMM_epi = df_all.query(" metric == 'LDDMM' and organ == 'epicardium' ")
df_chamfer_la = df_all.query(" metric == 'chamfer' and organ == 'la_endo' ")
df_LDDMM_la = df_all.query(" metric == 'LDDMM' and organ == 'la_endo' ")
df_chamfer_ra = df_all.query(" metric == 'chamfer' and organ == 'ra_endo' ")
df_LDDMM_ra = df_all.query(" metric == 'LDDMM' and organ == 'ra_endo' ")

Using vectorized built-in rank function of pandas. Leaving metric and organ columns for clarity and to not loose information of what this data is from.

In [38]:
ranked_chamfer = []
col_names = ["mean_rank_epi", "mean_rank_la", "mean_rank_ra"]
for i,df in enumerate([df_chamfer_epi, df_chamfer_la, df_chamfer_ra]):

    df_ranked = df.copy()
    df_ranked["rank"] = (
        df_ranked
        .groupby("patient")["value"]
        .rank(method="min", ascending=True)
    )
    df_ranked = df_ranked.groupby(["version"])["rank"].agg(mean="mean").reset_index()
    df_ranked = df_ranked.rename(columns={"mean": col_names[i]})
    ranked_chamfer.append(df_ranked)


In [43]:
df_ranked_all = reduce( lambda left, right: pd.merge(left, right, on="version", how="inner"), ranked_chamfer )
df_ranked_all

,version,mean_rank_epi,mean_rank_la,mean_rank_ra
0,89,2.5,2.3,2.3
1,100,2.4,2.4,2.4
2,114,1.1,1.3,1.3


In [46]:
df_long = df_ranked_all.melt(
    id_vars="version",        # keep version
    value_vars=["mean_rank_epi", "mean_rank_la", "mean_rank_ra"],  # columns to stack
    var_name="organ",         # name for the new “column identifier”
    value_name="mean_rank"    # name for the values
)
df_long.groupby("version")["mean_rank"].mean().reset_index()

,version,mean_rank
0,89,2.366667
1,100,2.400000
2,114,1.233333


Then if I have several tables like that, for several activation functions for example, I can put them togheter with versions VS activations, filing the table with ranks (or actual mean of chamfer distance instead of pure rank ...)

# Lipschitz layers VS activation functions

In [ ]:
# for these experiments, names are just like {version}-{experiment_name}-chamfer-{opt}.parquet
# retrieve all the wanted files
dfs = []

for file_path in Path("results/metrics").glob("*-LipAndAct*.parquet"):
    # extract version from filename, e.g. version_89.csv → 89
    version = file_path.stem.split("-")[0].split("_")[-1]
    # go fetch the specs file
    exp = "training_sweeps/" + file_path.stem.split("-")[1]
    with open(f"experiments/{exp}/version_{version}/hparams.json") as f:
        specs = json.load(f)
    lip_layers = specs["Network_specs"]["lipschitz_layers"]
    act = specs["Network_specs"]["activation"]
    df = pd.read_parquet(file_path)
    df["version"] = int(version)
    df["lip"] = str(lip_layers) # has to be a single value
    df["act"] = act
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all["value"] = df_all["value"].apply(parse_value)
df_all.drop(columns="version") # each version is a combination of lip and act, which I really want, I donìt actually need version number

,patient,organ,metric,value,lip,act
0,AF059,epicardium,chamfer,0.131684,"[0, 1, 2, 3, 4, 5, 6]",Tanh
1,AF059,la_endo,chamfer,0.137166,"[0, 1, 2, 3, 4, 5, 6]",Tanh
2,AF059,ra_endo,chamfer,0.229357,"[0, 1, 2, 3, 4, 5, 6]",Tanh
3,LEU_NORM_F017,epicardium,chamfer,0.081613,"[0, 1, 2, 3, 4, 5, 6]",Tanh
4,LEU_NORM_F017,la_endo,chamfer,0.103561,"[0, 1, 2, 3, 4, 5, 6]",Tanh
...,...,...,...,...,...,...
787,AF037,la_endo,LDDMM,0.001244,"[0, 1, 2, 3, 4, 5, 6]",GELU
788,AF037,ra_endo,LDDMM,0.003292,"[0, 1, 2, 3, 4, 5, 6]",GELU
789,LEU_NORM_0717,epicardium,LDDMM,0.014379,"[0, 1, 2, 3, 4, 5, 6]",GELU
790,LEU_NORM_0717,la_endo,LDDMM,0.003168,"[0, 1, 2, 3, 4, 5, 6]",GELU


In [ ]:
df_all["patient"].nunique()
df_all[["lip", "act"]].drop_duplicates().shape

(13, 2)

In [20]:
df = df_all.groupby(["organ", "metric", "lip", "act"])["value"].agg( 
    mean="mean",
    std="std"
).reset_index()
df

,organ,metric,lip,act,mean,std
0,epicardium,LDDMM,[-1],GELU,0.017681,0.021054
1,epicardium,LDDMM,[-1],Softplus,0.075649,0.173998
2,epicardium,LDDMM,[-1],Tanh,0.097028,0.125282
3,epicardium,LDDMM,"[0, 1, 2, 3, 4, 5, 6]",GELU,0.013076,0.014551
4,epicardium,LDDMM,"[0, 1, 2, 3, 4, 5, 6]",SiLU,0.008808,0.016015
...,...,...,...,...,...,...
73,ra_endo,chamfer,"[2, 3, 4, 5, 6]",SiLU,0.037500,0.017342
74,ra_endo,chamfer,"[2, 3, 4, 5, 6]",Softplus,0.043586,0.014512
75,ra_endo,chamfer,"[2, 3, 4, 5, 6]",Tanh,0.160764,0.059508
76,ra_endo,chamfer,"[4, 5, 6]",GELU,0.035130,0.014899


In [22]:
df_chamfer = df.query(" metric == 'chamfer' ").drop(columns="metric")
df_chamfer

,organ,lip,act,mean,std
13,epicardium,[-1],GELU,0.053962,0.022076
14,epicardium,[-1],Softplus,0.059933,0.017075
15,epicardium,[-1],Tanh,0.092244,0.059719
16,epicardium,"[0, 1, 2, 3, 4, 5, 6]",GELU,0.051078,0.022184
17,epicardium,"[0, 1, 2, 3, 4, 5, 6]",SiLU,0.040346,0.021216
18,epicardium,"[0, 1, 2, 3, 4, 5, 6]",Softplus,0.049817,0.017116
19,epicardium,"[0, 1, 2, 3, 4, 5, 6]",Tanh,0.134374,0.064919
20,epicardium,"[2, 3, 4, 5, 6]",GELU,0.046103,0.025139
21,epicardium,"[2, 3, 4, 5, 6]",SiLU,0.045406,0.023288
22,epicardium,"[2, 3, 4, 5, 6]",Softplus,0.054044,0.014471


In [25]:
tables = {}

for organ in df["organ"].unique():
    tables[organ] = (
        df[df["organ"] == organ]
        .pivot_table(
            index="act",      # rows
            columns="lip",     # columns
            values="mean",     # table values
            aggfunc="mean"     # redundant but explicit
        )
        .sort_index()
    )
# each is a matrix act x lip now, PER ORGAN
# tables["epicardium"]
# tables["la_endo"]
# tables["ra_endo"]

In [28]:
tables["epicardium"]

lip,[-1],"[0, 1, 2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]","[4, 5, 6]"
act,,,,
GELU,0.035821,0.032077,0.028090,0.026628
SiLU,NaN,0.024577,0.030870,0.027968
Softplus,0.067791,0.076934,0.036743,NaN
Tanh,0.094636,0.232191,0.210041,NaN
